Approach - 1:
- Using API calls
- Here we get all the bills raised for a specific year.
- We have a unique generalCourtNumber assigned for every year.
- Below block focuses on extracting bills raised in the current year (2025-26 with generalCourtNumber=194)

In [2]:
import requests
import pandas as pd

generalCourtNumber = 194 # for current year (2025-26)

url='https://malegislature.gov/api/GeneralCourts/194/Documents'
resp = requests.get(url)
if resp.status_code == 404:
    print("No data found.")

resp.raise_for_status()
data = resp.json()
res=[]
for i in data: 
    res.append({
                'GeneralCourtNumber': generalCourtNumber,
                'BillNumber':i.get('BillNumber'),
               'Title': i.get('Title')})
Bill_details= pd.DataFrame(res)
print(Bill_details)

      GeneralCourtNumber BillNumber  \
0                    194      H3444   
1                    194      H1283   
2                    194      H1399   
3                    194      H1282   
4                    194      H1932   
...                  ...        ...   
8699                 194      S2640   
8700                 194      S2641   
8701                 194      H4598   
8702                 194       None   
8703                 194       None   

                                                  Title  
0               An Act relative to s-license compliance  
1       An Act relative to real estate disclosure forms  
2     An Act relative to an individual Medicare mark...  
3     An Act relative to the Massachusetts Uniform C...  
4     An Act relative to the uniform voidable transa...  
...                                                 ...  
8699  An Act to increase access to disposable menstr...  
8700  An Act relative to menstrual product ingredien...  
8701  Ord

This block extracts Senate roll call votes for each recorded roll call votes.
rollCallNumber is unique for every roll call session recorded. (Note- A specific bill can have multiple roll call numbers assigned based on number of voting occurred for the bill before it is finally approved)


In [5]:
import requests
import pandas as pd

generalCourtNumber = 194 # for current year (2025-26)
branch = "Senate" #senate roll call votes
rollCallNumber = 56 #session for which votes are recorded

# Extract roll calls for given session
def get_rollcalls(generalCourtNumber, branch, rollCallNumber):
    url = f"https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Branches/{branch}/RollCalls/{rollCallNumber}"
    resp = requests.get(url)
    if resp.status_code == 404:
        print("No data found.")
        return None
    resp.raise_for_status()
    data = resp.json()
    return data


data = get_rollcalls(generalCourtNumber, branch, rollCallNumber)

# Extract member name from API for roll call votes 
# Create a dataframe storing final data
res=[]
for i in data.keys(): 
    votes = data.get(i, [])  
    if not votes:            
        continue
    if i not in [ 'Yeas','Nays','Absent']: 
        continue
    if len(data[i])==0: 
        continue
    x=0
    for j in data[i]:
        x+=1
        member_url= j['Details']
        # Handle case when unable to fetch member code or url
        if not member_url:
            res.append({
                        'RollCallNumber': rollCallNumber,
                        'Name': 'NA', 
                        'MemberCode': j.get('MemberCode', 'NA'), 
                        'Vote': i})
            continue
        resp_mem = requests.get(member_url)
        # Handle case when unable to fetch member code or url
        if resp_mem.status_code == 404:
            res.append({
                        'RollCallNumber': rollCallNumber,
                        'Name':'NA', 
                        'MemberCode': j.get('MemberCode', 'NA'),
                        'Vote':i})
            continue
        resp_mem.raise_for_status()
        data_mem = resp_mem.json()
        res.append({
                    'RollCallNumber': rollCallNumber,
                    'Name':data_mem['Name'], 
                    'MemberCode':j['MemberCode'], 
                    'Vote':i})

senate_votes = pd.DataFrame(res)
print(senate_votes)


    RollCallNumber                      Name MemberCode  Vote
0               56           Kelly A. Dooner       KAD0  Yeas
1               56             Bruce E. Tarr       BET0  Yeas
2               56       Patrick M. O'Connor        PMO  Yeas
3               56           Ryan C. Fattman       RCF0  Yeas
4               56           Peter J. Durant       PJD0  Yeas
5               56        Dylan A. Fernandes       DAF0  Nays
6               56  William J. Driscoll, Jr.       WJD0  Nays
7               56              Paul W. Mark       PWM0  Nays
8               56           Pavel M. Payano       PMP0  Nays
9               56          Robyn K. Kennedy       RKK0  Nays
10              56         Jacob R. Oliveira       JRO0  Nays
11              56               Liz Miranda       L M0  Nays
12              56         Barry R. Finegold       BRF0  Nays
13              56       Brendan P. Crighton       BPC0  Nays
14              56       Cynthia Stone Creem       CSC0  Nays
15      

In [ ]:
def get_bill_rollcall_map(generalCourtNumber, documentNumber):
    url = f'https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Documents/{documentNumber}/RollCalls'
    resp = requests.get(url)
    if resp.status_code == 404:
        print("No data found.")
    resp.raise_for_status()
    data = resp.json()
    res={}
    for i in data: 
        if i in res: res[i]= max(res[i], i.get('RollCallNumber'))
        else: res[i]= i.get('RollCallNumber')
        res.append({
                    'GeneralCourtNumber': generalCourtNumber,
                    'BillNumber': documentNumber,
                    'RollCallNumber': i.get('RollCallNumber')})
    Bill_rollcall_map= pd.DataFrame()
    return Bill_rollcall_map

bill_rollcall_map=get_bill_rollcall_map(194, 'S2538')

print(bill_rollcall_map)


   GeneralCourtNumber BillNumber  RollCallNumber
0                 194      S2538              55
1                 194      S2538              56
2                 194      S2538              57


As a bill can have multiple roll call numbers, we need to take the highest rollcallno: considering it as the latest record for a bill after all amendments 
(If all teammates are aligned on this approach, will implement this step in final data retrieval)

In [34]:
#final join 
result1= pd.merge(Bill_details, bill_rollcall_map, how='inner', on='BillNumber', suffixes=('', '_drop'))
result2= pd.merge(result1, senate_votes, how='inner', on='RollCallNumber', suffixes=('', '_drop'))
denorm_data = result2[[c for c in result2.columns if not c.endswith('_drop')]]
denorm_data.loc[:, 'Vote']= denorm_data['Vote'].replace({'Yeas':'Y', 'Nays':'N'})
print(denorm_data.head())

   GeneralCourtNumber BillNumber  \
0                 194      S2538   
1                 194      S2538   
2                 194      S2538   
3                 194      S2538   
4                 194      S2538   

                                               Title  RollCallNumber  \
0  An Act strengthening health care protections i...              56   
1  An Act strengthening health care protections i...              56   
2  An Act strengthening health care protections i...              56   
3  An Act strengthening health care protections i...              56   
4  An Act strengthening health care protections i...              56   

                  Name MemberCode Vote  
0      Kelly A. Dooner       KAD0    Y  
1        Bruce E. Tarr       BET0    Y  
2  Patrick M. O'Connor        PMO    Y  
3      Ryan C. Fattman       RCF0    Y  
4      Peter J. Durant       PJD0    Y  


Approach-2
- Using PDF from Roll Calls / Amendments sections from Mass legislature website
- https://malegislature.gov/Bills/194/S2538/RollCall 
- https://malegislature.gov/Bills/194/S2538/Amendments/Senate

In [ ]:
import requests
from pdf2image import convert_from_bytes
import pytesseract
import pandas as pd

def extract_rollcall_text(pdf_url):
    response = requests.get(pdf_url)
    response.raise_for_status()

    # Convert PDF to images
    images = convert_from_bytes(response.content)

    # Using Tesseract to extract text from images
    text = ""
    for img in images:
        custom_config = r'--oem 3 --psm 11'
        text += pytesseract.image_to_string(img, config=custom_config)
    return text

def parse_rollcall_text(raw_text, bill_no):
    results = []
    prev=''
    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue
        if 'YEA' in line or 'NAY' in line or 'PRESENT' in line or 'ABSENT' in line:
            if 'YEA' in line: prev='YEA'
            elif 'NAY' in line: prev='NAY'
            elif 'PRESENT' in line: prev='PRESENT'
            else: prev= 'ABSENT'
        elif prev=='': continue
        else:
            results.append({"bill no:": bill_no, "name": line.replace(prev,'').strip(), "vote": prev})
    return results


pdf_url = "https://malegislature.gov/RollCall/194/SenateRollCall56.pdf"
text = extract_rollcall_text(pdf_url)
data = parse_rollcall_text(text, 'S2538')
df = pd.DataFrame(data)
df['vote'] = df['vote'].replace({'YEA': 'Y', 'NAY': 'N'})
# Handling suffix in names like . / -5
df['name'] = df['name'].str.replace(r'\.\s*[-–—]?\s*\d*.*$', '', regex=True).str.strip()
print(df.head())
   


  bill no:                 name vote
0    S2538      Dooner, Kelly A    Y
1    S2538  O'Connor, Patrick M    Y
2    S2538      Durant, Peter J    Y
3    S2538        Tarr, Bruce E    Y
4    S2538      Fattman, Ryan C    Y
